In [ ]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torch.backends.cudnn as cudnn
import os
import json
from models import (
    Encoder,
    DecoderWithRNN,
    DecoderWithAttention,
    DecoderWithAdaptiveAttention,
)
from datasets import *
from solver import train, validate
from utils import save_checkpoint, adjust_learning_rate
from visualization import TrainingVisualizer

In [ ]:
# ==================== 配置参数 ====================
cfg = {
    # 数据参数
    "data_folder": "data/coco2014",  # 数据文件夹路径
    "data_name": "coco_5_cap_per_img_5_min_word_freq",  # 数据集基础名称
    
    # 模型参数
    "embed_dim": 512,  # 词嵌入维度
    "attention_dim": 512,  # 注意力层维度
    "decoder_dim": 512,  # 解码器RNN隐藏层维度
    "dropout": 0.5,  # Dropout比率
    "device": torch.device(
        "cuda:0" if torch.cuda.is_available() else "cpu"
    ),  # 计算设备
    
    # 训练参数
    "start_epoch": 0,  # 起始epoch
    "epochs": 20,  # 训练总epoch数
    "epochs_since_improvement": 0,  # 自上次改进以来的epoch数
    "batch_size": 32,  # 批大小
    "workers": 1,  # 数据加载器工作线程数
    "encoder_lr": 1e-4,  # 编码器学习率
    "decoder_lr": 4e-4,  # 解码器学习率
    "grad_clip": 5.0,  # 梯度裁剪阈值
    "alpha_c": 1.0,  # doubly stochastic attention正则化参数
    "best_bleu4": 0.0,  # 当前最佳BLEU-4分数
    "print_freq": 100,  # 每多少批次打印一次训练状态
    "fine_tune_encoder": False,  # 是否微调编码器
    "checkpoint": None,  # 检查点路径，None表示从头训练
    "attention": True,  # 是否使用注意力机制
    "adaptive_attention": True,  # 是否使用自适应注意力
}

In [ ]:
# ==================== 初始化设置 ====================
cudnn.benchmark = True  # 仅当模型输入大小固定时设置为True

# 加载词汇表
word_map_file = os.path.join(
    cfg["data_folder"], "WORDMAP_" + cfg["data_name"] + ".json"
)
with open(word_map_file, "r") as j:
    word_map = json.load(j)
cfg["vocab_size"] = len(word_map)

print(f"\n{'='*70}")
print(f"Image Captioning Training")
print(f"{'='*70}")
print(f"Device: {cfg['device']}")
print(f"Vocabulary Size: {cfg['vocab_size']}")
print(f"Attention: {cfg['attention']}")
print(f"Adaptive Attention: {cfg['adaptive_attention']}")
print(f"{'='*70}\n")

In [ ]:
# ==================== 模型初始化 ====================
if cfg["checkpoint"] is None:
    # 从头开始训练
    print("🔧 Initializing models from scratch...")
    
    # 初始化编码器
    encoder = Encoder()
    encoder.fine_tune(cfg["fine_tune_encoder"])
    encoder_optimizer = (
        torch.optim.AdamW(
            params=filter(lambda p: p.requires_grad, encoder.parameters()),
            lr=cfg["encoder_lr"],
        )
        if cfg["fine_tune_encoder"]
        else None
    )
    
    # 初始化解码器 - 修正拼写错误
    if not cfg["attention"]:
        print("  → Using DecoderWithRNN (no attention)")
        decoder = DecoderWithRNN(cfg)
    elif cfg["attention"] and not cfg["adaptive_attention"]:
        print("  → Using DecoderWithAttention (basic attention)")
        decoder = DecoderWithAttention(cfg)
    else:
        print("  → Using DecoderWithAdaptiveAttention (adaptive attention)")
        decoder = DecoderWithAdaptiveAttention(cfg)
    
    decoder_optimizer = torch.optim.AdamW(
        params=filter(lambda p: p.requires_grad, decoder.parameters()),
        lr=cfg["decoder_lr"],
    )
    
    print("✓ Models initialized successfully\n")
    
else:
    # 从检查点恢复训练
    print(f"🔄 Loading checkpoint from: {cfg['checkpoint']}")
    checkpoint = torch.load(cfg["checkpoint"])
    
    cfg["start_epoch"] = checkpoint["epoch"] + 1
    cfg["epochs_since_improvement"] = checkpoint["epochs_since_improvement"]
    cfg["best_bleu4"] = checkpoint["bleu-4"]
    
    encoder = checkpoint["encoder"]
    encoder_optimizer = checkpoint["encoder_optimizer"]
    decoder = checkpoint["decoder"]
    decoder_optimizer = checkpoint["decoder_optimizer"]
    
    # 如果需要微调编码器但检查点中没有编码器优化器
    if cfg["fine_tune_encoder"] is True and encoder_optimizer is None:
        encoder.fine_tune(cfg["fine_tune_encoder"])
        encoder_optimizer = torch.optim.AdamW(
            params=filter(lambda p: p.requires_grad, encoder.parameters()),
            lr=cfg["encoder_lr"],
        )
    
    print(f"✓ Checkpoint loaded successfully")
    print(f"  → Resuming from epoch {cfg['start_epoch']}")
    print(f"  → Best BLEU-4 so far: {cfg['best_bleu4']}\n")

# 移动模型到GPU
decoder = decoder.to(cfg["device"])
encoder = encoder.to(cfg["device"])

# ==================== 损失函数 ====================
criterion = nn.CrossEntropyLoss().to(cfg["device"])

# ==================== 数据加载器 ====================
print("📁 Loading datasets...")

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], 
    std=[0.229, 0.224, 0.225]
)

train_loader = torch.utils.data.DataLoader(
    CaptionDataset(
        cfg["data_folder"],
        cfg["data_name"],
        "TRAIN",
        transform=transforms.Compose([normalize]),
    ),
    batch_size=cfg["batch_size"],
    shuffle=True,
    num_workers=cfg["workers"],
    pin_memory=True,
)

val_loader = torch.utils.data.DataLoader(
    CaptionDataset(
        cfg["data_folder"],
        cfg["data_name"],
        "VAL",
        transform=transforms.Compose([normalize]),
    ),
    batch_size=cfg["batch_size"],
    shuffle=True,
    num_workers=cfg["workers"],
    pin_memory=True,
)

print(f"✓ Datasets loaded")
print(f"  → Training batches: {len(train_loader)}")
print(f"  → Validation batches: {len(val_loader)}\n")

# ==================== 可视化器初始化 ====================
experiment_name = f"{cfg['data_name']}_adaptive_{cfg['adaptive_attention']}"
visualizer = TrainingVisualizer(
    save_dir='./visualizations',
    experiment_name=experiment_name
)

# 保存配置
visualizer.save_config(cfg)
print(f"📊 Visualizer initialized")
print(f"  → Results will be saved to: {visualizer.exp_dir}\n")

# ==================== 训练循环 ====================
print(f"{'='*70}")
print(f"Starting Training")
print(f"{'='*70}\n")

try:
    for epoch in range(cfg["start_epoch"], cfg["epochs"]):
        # 将当前epoch保存到cfg中供可视化使用
        cfg['current_epoch'] = epoch
        
        # 学习率衰减策略
        if cfg["epochs_since_improvement"] == 20:
            print("\n" + "="*70)
            print("⛔ No improvement for 20 epochs. Early stopping triggered.")
            print("="*70 + "\n")
            break
            
        if cfg["epochs_since_improvement"] > 0 and cfg["epochs_since_improvement"] % 8 == 0:
            print("\n" + "─"*70)
            print(f"⚡ Learning Rate Decay")
            print(f"   No improvement for {cfg['epochs_since_improvement']} epochs")
            print(f"   Reducing learning rate by 0.8x")
            adjust_learning_rate(decoder_optimizer, 0.8)
            if cfg["fine_tune_encoder"]:
                adjust_learning_rate(encoder_optimizer, 0.8)
            print("─"*70 + "\n")

        # ==================== 训练阶段 ====================
        print(f"\n{'='*70}")
        print(f"Epoch {epoch + 1}/{cfg['epochs']} - Training Phase")
        print(f"{'='*70}")
        
        train(
            train_loader=train_loader,
            encoder=encoder,
            decoder=decoder,
            criterion=criterion,
            encoder_optimizer=encoder_optimizer,
            decoder_optimizer=decoder_optimizer,
            epoch=epoch,
            cfg=cfg,
            visualizer=visualizer
        )

        # ==================== 验证阶段 ====================
        print(f"\n{'='*70}")
        print(f"Epoch {epoch + 1}/{cfg['epochs']} - Validation Phase")
        print(f"{'='*70}")
        
        recent_bleu4 = validate(
            val_loader=val_loader,
            encoder=encoder,
            decoder=decoder,
            criterion=criterion,
            word_map=word_map,
            cfg=cfg,
            visualizer=visualizer
        )

        # ==================== 检查改进 ====================
        is_best = recent_bleu4 > cfg["best_bleu4"]
        cfg["best_bleu4"] = max(recent_bleu4, cfg["best_bleu4"])
        
        if not is_best:
            cfg["epochs_since_improvement"] += 1
            print(f"\n{'─'*70}")
            print(f"⚠  No improvement. Epochs since improvement: {cfg['epochs_since_improvement']}")
            print(f"   Current BLEU-4: {recent_bleu4}")
            print(f"   Best BLEU-4: {cfg['best_bleu4']}")
            print(f"{'─'*70}\n")
        else:
            cfg["epochs_since_improvement"] = 0
            print(f"\n{'─'*70}")
            print(f"🎉 New Best Model!")
            print(f"   Previous best: {cfg['best_bleu4'] - (recent_bleu4 - cfg['best_bleu4'])}")
            print(f"   New best: {cfg['best_bleu4']}")
            print(f"   Improvement: +{recent_bleu4 - (cfg['best_bleu4'] - (recent_bleu4 - cfg['best_bleu4']))}")
            print(f"{'─'*70}\n")

        # ==================== 保存检查点 ====================
        save_checkpoint(
            cfg["data_name"],
            epoch,
            cfg["epochs_since_improvement"],
            encoder,
            decoder,
            encoder_optimizer,
            decoder_optimizer,
            recent_bleu4,
            is_best,
        )

        # ==================== 生成可视化 ====================
        print(f"📊 Generating visualizations for epoch {epoch + 1}...")
        visualizer.plot_all()
        visualizer.save_metrics()
        
        # 每5个epoch生成一次详细报告
        if (epoch + 1) % 5 == 0:
            visualizer.generate_report()
            print(f"✓ Training report generated")
        
        print(f"✓ Visualizations updated\n")

except KeyboardInterrupt:
    print("\n\n" + "="*70)
    print("⚠  Training interrupted by user")
    print("="*70)
    print("Saving current state...\n")
    
    # 保存中断时的状态
    save_checkpoint(
        cfg["data_name"],
        cfg.get('current_epoch', cfg['start_epoch']),
        cfg["epochs_since_improvement"],
        encoder,
        decoder,
        encoder_optimizer,
        decoder_optimizer,
        cfg["best_bleu4"],
        False,
    )
    print("✓ Checkpoint saved\n")

except Exception as e:
    print("\n\n" + "="*70)
    print("❌ Training failed with error:")
    print("="*70)
    print(f"{type(e).__name__}: {e}\n")
    raise

finally:
    # ==================== 训练结束总结 ====================
    print(f"\n{'='*70}")
    print(f"Training Session Completed")
    print(f"{'='*70}")
    print(f"Total epochs trained: {cfg.get('current_epoch', cfg['start_epoch']) - cfg['start_epoch'] + 1}")
    print(f"Best BLEU-4 achieved: {cfg['best_bleu4']}")
    print(f"{'='*70}\n")
    
    # 生成最终报告
    print("📊 Generating final training report...")
    visualizer.generate_report()
    visualizer.save_metrics()
    
    print(f"\n✅ All results saved to: {visualizer.exp_dir}")
    print(f"\nGenerated files:")
    print(f"  📈 loss_curves.png")
    print(f"  📈 accuracy_curves.png")
    print(f"  📈 bleu_curve.png")
    print(f"  📈 learning_rate.png")
    print(f"  📈 attention_statistics.png")
    print(f"  📈 time_statistics.png")
    print(f"  📈 comprehensive_dashboard.png")
    print(f"  📄 metrics.json")
    print(f"  📄 training_report.txt")
    print(f"  📄 config.json")
    print(f"\n{'='*70}\n")